In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Fixer l'aléatoire pour avoir exactement les mêmes données si on relance
np.random.seed(42)
random.seed(42)

# 1. Catalogue produits (Secteur Beauté / Luxe)
catalogue = [
    ("Sérum Éclat Intense 30ml", "Soins Visage", 85.0),
    ("Crème Hydratante Velours", "Soins Visage", 68.0),
    ("Huile Démaquillante Douce", "Soins Visage", 34.0),
    ("Contour des Yeux Régénérant", "Soins Visage", 55.0),
    ("Eau de Parfum Nuit d'Ambre 50ml", "Parfums", 115.0),
    ("Eau de Parfum Rose Sauvage 100ml", "Parfums", 145.0),
    ("Brume Parfumée Corps & Cheveux", "Parfums", 48.0),
    ("Rouge à Lèvres Satiné - Carmin", "Maquillage", 38.0),
    ("Mascara Volume Absolu", "Maquillage", 32.0),
    ("Poudre Lumière Sublimatrice", "Maquillage", 52.0),
    ("Coffret Découverte Rituel Beauté", "Coffrets", 160.0),
    ("Coffret Duo Parfums Signature", "Coffrets", 195.0)
]

# 2. Création de 2 500 clients avec des profils d'achat différents
n_customers = 2500
customer_ids = [f"CUST_{10000 + i}" for i in range(n_customers)]
countries = ["France", "Belgique", "Suisse", "Luxembourg", "Canada"]
country_weights = [0.65, 0.12, 0.10, 0.05, 0.08]
customer_country = {cid: np.random.choice(countries, p=country_weights) for cid in customer_ids}

# 3. Génération de 12 000 lignes de commandes entre 2024 et 2026
rows = []
start_date = datetime(2024, 1, 1)
end_date = datetime(2026, 10, 31)
days_range = (end_date - start_date).days

for i in range(12000):
    order_id = f"ORD_{50000 + i}"
    # Certains clients achètent beaucoup plus que d'autres (effet Pareto)
    cid = np.random.choice(customer_ids, p=np.random.dirichlet(np.ones(n_customers) * 0.6))

    # Date avec saisonnalité (plus d'achats en novembre/décembre et février)
    random_days = random.randint(0, days_range)
    order_date = start_date + timedelta(days=random_days)
    if order_date.month not in [2, 11, 12] and random.random() < 0.35:
        order_date = order_date.replace(month=random.choice([2, 11, 12]), day=random.randint(1, 28))

    prod, cat, price = random.choice(catalogue)
    qty = np.random.choice([1, 2, 3, 4], p=[0.65, 0.22, 0.09, 0.04])
    country = customer_country[cid]

    rows.append([cid, order_id, order_date.strftime("%Y-%m-%d"), prod, cat, price, qty, country])

df = pd.DataFrame(rows, columns=["customer_id", "order_id", "date", "product", "category", "price", "quantity", "country"])

# 4. Injection volontaire d'anomalies réalistes (pour l'étape de nettoyage !)
# a) Retours produits (quantités négatives sur 3% des commandes)
return_idx = df.sample(frac=0.03, random_state=42).index
df.loc[return_idx, "quantity"] = -df.loc[return_idx, "quantity"]

# b) Valeurs manquantes sur customer_id et country
df.loc[df.sample(frac=0.015, random_state=1).index, "customer_id"] = np.nan
df.loc[df.sample(frac=0.01, random_state=2).index, "country"] = np.nan

# c) Fautes de frappe sur les pays et espaces inutiles
typo_idx = df.sample(frac=0.02, random_state=3).index
df.loc[typo_idx, "country"] = df.loc[typo_idx, "country"].replace({"France": "france ", "Belgique": "BE", "Suisse": "suisse"})

# d) Formats de dates incohérents (DD/MM/YYYY mélangé avec YYYY-MM-DD)
date_idx = df.sample(frac=0.05, random_state=4).index
df.loc[date_idx, "date"] = pd.to_datetime(df.loc[date_idx, "date"]).dt.strftime("%d/%m/%Y")

# e) Ajout de 150 lignes en doublon parfait
duplicates = df.sample(n=150, random_state=5)
df = pd.concat([df, duplicates], ignore_index=True)

# Sauvegarde dans le dossier data
df.to_csv("raw_ecommerce_sales.csv", index=False)
print(f"Fichier généré avec succès ! Nombre total de lignes : {len(df)}")
df.head(10)

Fichier généré avec succès ! Nombre total de lignes : 12150


,customer_id,order_id,date,product,category,price,quantity,country
0,CUST_12023,ORD_50000,08/11/2024,Contour des Yeux Régénérant,Soins Visage,55.0,2,France
1,CUST_10523,ORD_50001,2024-10-12,Coffret Découverte Rituel Beauté,Coffrets,160.0,1,Suisse
2,CUST_10070,ORD_50002,2024-06-27,Sérum Éclat Intense 30ml,Soins Visage,85.0,1,Suisse
3,CUST_10169,ORD_50003,2024-02-17,Poudre Lumière Sublimatrice,Maquillage,52.0,1,France
4,CUST_12220,ORD_50004,2024-02-24,Mascara Volume Absolu,Maquillage,32.0,1,Belgique
5,CUST_11242,ORD_50005,2025-02-11,Coffret Duo Parfums Signature,Coffrets,195.0,1,Suisse
6,CUST_12430,ORD_50006,2026-12-09,Sérum Éclat Intense 30ml,Soins Visage,85.0,1,France
7,CUST_10666,ORD_50007,2024-11-22,Coffret Duo Parfums Signature,Coffrets,195.0,2,France
8,CUST_11590,ORD_50008,2026-02-07,Eau de Parfum Rose Sauvage 100ml,Parfums,145.0,1,Belgique
9,CUST_11679,ORD_50009,2024-02-12,Eau de Parfum Rose Sauvage 100ml,Parfums,145.0,1,Belgique


In [2]:
# Chargement des données brutes
df_clean = pd.read_csv("raw_ecommerce_sales.csv")
print(f"Lignes initiales : {len(df_clean)}")

# 1. Suppression des doublons parfaits
doublons = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()
print(f"- Doublons supprimés : {doublons}")

# 2. Gestion des valeurs manquantes
# a) Pour customer_id : indispensable pour le CRM, on supprime les lignes sans ID client
sans_client = df_clean["customer_id"].isna().sum()
df_clean = df_clean.dropna(subset=["customer_id"])
print(f"- Lignes sans customer_id supprimées : {sans_client}")

# b) Pour country : on essaie d'abord de retrouver le pays du client via ses autres commandes
client_pays_map = df_clean.dropna(subset=["country"]).set_index("customer_id")["country"].to_dict()
df_clean["country"] = df_clean["country"].fillna(df_clean["customer_id"].map(client_pays_map))
df_clean["country"] = df_clean["country"].fillna("Inconnu")

# 3. Standardisation des noms de pays (espaces et abréviations)
df_clean["country"] = df_clean["country"].str.strip().str.capitalize()
mapping_pays = {
    "Be": "Belgique",
    "Fr": "France",
    "Ch": "Suisse"
}
df_clean["country"] = df_clean["country"].replace(mapping_pays)

# 4. Uniformisation des dates (format mixte -> datetime propre)
df_clean["date"] = pd.to_datetime(df_clean["date"], format="mixed", dayfirst=True)

# 5. Traitement métier des retours & Feature Engineering
df_clean["is_return"] = df_clean["quantity"] < 0
df_clean["total_amount"] = df_clean["price"] * df_clean["quantity"]
df_clean["year_month"] = df_clean["date"].dt.to_period("M").astype(str)
df_clean["day_of_week"] = df_clean["date"].dt.day_name()

print(f"\nNettoyage terminé ! Nombre de lignes propres : {len(df_clean)}")
print(f"Dont retours produits identifiés : {df_clean['is_return'].sum()}")
print("\nRépartition propre par pays :")
print(df_clean["country"].value_counts())

# Sauvegarde du fichier prêt pour SQL et Power BI
df_clean.to_csv("clean_ecommerce_sales.csv", index=False)

Lignes initiales : 12150
- Doublons supprimés : 150
- Lignes sans customer_id supprimées : 180

Nettoyage terminé ! Nombre de lignes propres : 11820
Dont retours produits identifiés : 351

Répartition propre par pays :
country
France        7765
Belgique      1355
Suisse        1211
Canada         934
Luxembourg     555
Name: count, dtype: int64


In [3]:
# On travaille uniquement sur les ventes réelles (hors retours) pour la valeur client
sales_only = df_clean[df_clean["is_return"] == False].copy()

# Date de référence = lendemain de la dernière commande du dataset
snapshot_date = sales_only["date"].max() + pd.Timedelta(days=1)

# Calcul des métriques R, F, M par client
rfm = sales_only.groupby("customer_id").agg(
    recency=("date", lambda x: (snapshot_date - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("total_amount", "sum"),
    country=("country", "first")
).reset_index()

# Attribution des scores de 1 à 5 via les quintiles
# Attention : pour Recency, un petit nombre de jours = meilleure note (5)
rfm["R_score"] = pd.qcut(rfm["recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_score"] = pd.qcut(rfm["monetary"], 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm["RFM_score"] = rfm["R_score"].astype(str) + rfm["F_score"].astype(str) + rfm["M_score"].astype(str)

# Définition des segments marketing métier
def assign_segment(row):
    r, f, m = row["R_score"], row["F_score"], row["M_score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "1. Champions (VIP)"
    elif f >= 3 and m >= 3:
        return "2. Clients Fidèles"
    elif r >= 4 and f <= 2:
        return "3. Nouveaux / À Fort Potentiel"
    elif r <= 2 and (f >= 3 or m >= 3):
        return "4. À Risque (À relancer d'urgence)"
    else:
        return "5. Clients Endormis / Perdus"

rfm["segment"] = rfm.apply(assign_segment, axis=1)

# Résumé business par segment
resume_rfm = rfm.groupby("segment").agg(
    nb_clients=("customer_id", "count"),
    ca_total=("monetary", "sum"),
    panier_cumule_moyen=("monetary", "mean"),
    recence_moyenne_jours=("recency", "mean")
).reset_index()

resume_rfm["part_clients_%"] = (resume_rfm["nb_clients"] / resume_rfm["nb_clients"].sum() * 100).round(1)
resume_rfm["part_ca_%"] = (resume_rfm["ca_total"] / resume_rfm["ca_total"].sum() * 100).round(1)

print("=== SYNTHÈSE SEGMENTATION RFM ===")
display(resume_rfm)

# Export de la table clients segmentée
rfm.to_csv("rfm_customers.csv", index=False)

=== SYNTHÈSE SEGMENTATION RFM ===


,segment,nb_clients,ca_total,panier_cumule_moyen,recence_moyenne_jours,part_clients_%,part_ca_%
0,1. Champions (VIP),381,386479.0,1014.380577,47.165354,15.4,26.0
1,2. Clients Fidèles,841,666001.0,791.915577,220.662307,34.0,44.8
2,3. Nouveaux / À Fort Potentiel,271,105821.0,390.483395,50.468635,11.0,7.1
3,4. À Risque (À relancer d'urgence),229,114766.0,501.161572,403.751092,9.3,7.7
4,5. Clients Endormis / Perdus,752,215009.0,285.916223,360.683511,30.4,14.4
